## working perfect in 5 mins chart

In [2]:
import pandas as pd
import numpy as np
from datetime import timedelta
import ta.volatility

class SuperV3SignalGenerator:
    def __init__(self, file_path, output_file):
        self.file_path = file_path
        self.output_file = output_file
        self.data = self.load_data()
        self.setup_parameters()

    def load_data(self):
        return pd.read_csv(self.file_path)

    def setup_parameters(self):
        # Define parameters
        self.dist1 = 14
        self.dist2 = 21

    def calculate_indicators(self):
        df = self.data
        
        # Calculate highest and lowest values over dist1 and dist2 periods
        df['hhb1'] = df['high'].rolling(window=self.dist1, center=True).max()
        df['llb1'] = df['low'].rolling(window=self.dist1, center=True).min()
        df['hhb'] = df['high'].rolling(window=self.dist2, center=True).max()
        df['llb'] = df['low'].rolling(window=self.dist2, center=True).min()
        
        # Calculate ATR
        df['atr'] = ta.volatility.average_true_range(df['high'], df['low'], df['close'], window=50)

    def generate_signals(self):
        df = self.data

        # Initialize b1, b2, b3, b4
        df['b1'] = np.nan
        df['b2'] = np.nan
        df['b3'] = np.nan
        df['b4'] = np.nan

        # Fill b1, b2, b3, b4 based on conditions
        df.loc[df['high'] == df['hhb'], 'b1'] = df['high'] + df['atr']
        df.loc[df['low'] == df['llb'], 'b2'] = df['low'] - df['atr']
        df.loc[df['high'] == df['hhb1'], 'b3'] = df['high'] + df['atr'] / 2
        df.loc[df['low'] == df['llb1'], 'b4'] = df['low'] - df['atr'] / 2

        # Generate signals
        conditions = [
            (df['b1'].notna() & df['b3'].notna(), "strong sell"),
            (df['b1'].notna() & df['b3'].isna(), "sell"),
            (df['b1'].isna() & df['b3'].notna(), "minor sell or exit buy"),
            (df['b2'].notna() & df['b4'].notna(), "strong buy"),
            (df['b2'].notna() & df['b4'].isna(), "buy"),
            (df['b2'].isna() & df['b4'].notna(), "minor buy or exit sell")
        ]

        # Apply conditions to generate signals
        for condition, signal in conditions:
            df.loc[condition, 'signal'] = signal

    def adjust_timezones(self):
        df = self.data
        df['UTC'] = pd.to_datetime(df['datetime']) + timedelta(hours=5)
        df['GMT'] = df['UTC'] + timedelta(hours=2)

    def save_output(self):
        self.data.to_csv(self.output_file, index=False)

    def run(self):
        self.calculate_indicators()
        self.generate_signals()
        self.adjust_timezones()
        self.save_output()



In [ ]:
# Example usage
file_path = '../common/MachineLearningModel/output/five_mins/EURUSD_5_Min_testing.csv'
output_file = '../common/MachineLearningModel/output/outputsupersignalV3Eurusd_1.csv'

processor = SuperV3SignalGenerator(file_path, output_file)
processor.run()
